# 02 — From Spinor to Bloch Sphere

A single-qubit state vector is called a **spinor**.  The Bloch sphere is the geometric way to see it.

This notebook shows:
- how the spinor maps to a point on the sphere
- why **global phase is invisible** (it cancels in every observable)
- how antipodal points encode orthogonal states
- how measurement probabilities read off from the $z$-coordinate

---

## The spinor

A normalised single-qubit state:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad |\alpha|^2 + |\beta|^2 = 1$$

Using the global phase freedom we can always write:

$$|\psi\rangle = \cos\!\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\!\frac{\theta}{2}|1\rangle$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import draw_bloch_sphere, plot_bloch_like_vector
from helpers.display_utils import show_state_vector, show_info_table, format_complex_pair, assert_state_close
setup_notebook()


## The spinor → Bloch vector map

Given amplitudes $(\alpha, \beta)$, the Bloch vector is:

$$x = 2\,\text{Re}(\alpha^*\beta), \quad y = 2\,\text{Im}(\alpha^*\beta), \quad z = |\alpha|^2 - |\beta|^2$$

This gives a one-to-one map between **physical** (ray) states and points on the unit sphere.

> **In production code use `rqm-core`'s `Spinor.to_bloch()`.  
> The formula below is illustrative only.**


In [ ]:
def spinor_to_bloch(alpha, beta):
    """Convert a normalised spinor to a Bloch vector.
    NOTE: use rqm-core in production.
    """
    alpha, beta = complex(alpha), complex(beta)
    norm = np.sqrt(abs(alpha)**2 + abs(beta)**2)
    alpha, beta = alpha / norm, beta / norm
    x = 2 * (alpha.conjugate() * beta).real
    y = 2 * (alpha.conjugate() * beta).imag
    z = abs(alpha)**2 - abs(beta)**2
    return np.array([x, y, z])

# Standard basis states
states = {
    r"|0⟩": (1, 0),
    r"|1⟩": (0, 1),
    r"|+⟩": (1/np.sqrt(2), 1/np.sqrt(2)),
    r"|-⟩": (1/np.sqrt(2), -1/np.sqrt(2)),
    r"|+i⟩": (1/np.sqrt(2), 1j/np.sqrt(2)),
    r"|-i⟩": (1/np.sqrt(2), -1j/np.sqrt(2)),
}

print(f"{'State':<8} {'x':>8} {'y':>8} {'z':>8}")
print("-" * 38)
for label, (a, b) in states.items():
    bv = spinor_to_bloch(a, b)
    print(f"{label:<8} {bv[0]:>8.4f} {bv[1]:>8.4f} {bv[2]:>8.4f}")


## Global phase invariance

A **global phase** $e^{i\varphi}$ multiplies both amplitudes by the same complex factor.  
Because it cancels in every observable, it is physically undetectable.

$$e^{i\varphi}|\psi\rangle \xrightarrow{\text{Bloch}} \text{same point}$$

Let's verify this computationally.


In [ ]:
alpha0, beta0 = 1/np.sqrt(3), np.sqrt(2/3) * np.exp(1j * np.pi / 4)
bv_orig = spinor_to_bloch(alpha0, beta0)

print("Original state:")
print(f"  {format_complex_pair(alpha0, beta0)}")
print(f"  Bloch vector: ({bv_orig[0]:.4f}, {bv_orig[1]:.4f}, {bv_orig[2]:.4f})")
print()

# Apply three different global phases
phases = [np.pi / 6, np.pi / 2, np.pi]
for phi in phases:
    a_rot = np.exp(1j * phi) * alpha0
    b_rot = np.exp(1j * phi) * beta0
    bv_rot = spinor_to_bloch(a_rot, b_rot)
    diff = np.linalg.norm(bv_rot - bv_orig)
    print(f"  Phase e^(i·{phi:.2f}π/…)  →  Bloch diff = {diff:.2e}  {'✓ same' if diff < 1e-10 else '✗ different'}")

print()
print("Conclusion: global phase is invisible on the Bloch sphere.")


## Visualise the standard states

All six cardinal states lie on the principal axes of the Bloch sphere.


In [ ]:
colour_map = {
    r"|0⟩": "green", r"|1⟩": "red",
    r"|+⟩": "steelblue", r"|-⟩": "darkorange",
    r"|+i⟩": "violet", r"|-i⟩": "sienna",
}

vectors = [(spinor_to_bloch(a, b), lbl, colour_map[lbl]) for lbl, (a, b) in states.items()]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
draw_bloch_sphere(ax=ax, vectors=vectors, title="Standard basis states on the Bloch sphere")
plt.tight_layout()
plt.show()


## Antipodal states are orthogonal

If $|\psi\rangle$ maps to $\vec{r}$, its antipodal state $|-\psi\rangle$ maps to $-\vec{r}$.  
These two states are **orthogonal**: $\langle\psi|-\psi\rangle = 0$.

| Pair | Bloch direction | Meaning |
|---|---|---|
| $|0\rangle$, $|1\rangle$ | $+z$, $-z$ | computational basis |
| $|+\rangle$, $|-\rangle$ | $+x$, $-x$ | X eigenstates |
| $|{+i}\rangle$, $|{-i}\rangle$ | $+y$, $-y$ | Y eigenstates |


In [ ]:
# Verify orthogonality of antipodal pairs
def inner_product(state_a, state_b):
    a, b = np.asarray(state_a, dtype=complex), np.asarray(state_b, dtype=complex)
    return np.dot(a.conjugate(), b)

antipodal_pairs = [
    (r"|0⟩", r"|1⟩"),
    (r"|+⟩", r"|-⟩"),
    (r"|+i⟩", r"|-i⟩"),
]

print("Antipodal orthogonality check:")
for a_lbl, b_lbl in antipodal_pairs:
    sa = np.array(states[a_lbl], dtype=complex) / np.linalg.norm(states[a_lbl])
    sb = np.array(states[b_lbl], dtype=complex) / np.linalg.norm(states[b_lbl])
    ip = inner_product(sa, sb)
    print(f"  ⟨{a_lbl}|{b_lbl}⟩ = {ip.real:.6f} + {ip.imag:.6f}i  ({'✓ orthogonal' if abs(ip) < 1e-10 else '✗'})")


## Measurement probabilities

From any Bloch vector $(x, y, z)$:

$$P(|0\rangle) = \frac{1 + z}{2}, \qquad P(|1\rangle) = \frac{1 - z}{2}$$

The $z$-coordinate encodes the computational-basis measurement outcome distribution.


In [ ]:
print(f"{'State':<8} {'z':>6} {'P(|0⟩)':>10} {'P(|1⟩)':>10}")
print("-" * 38)
for label, (a, b) in states.items():
    bv = spinor_to_bloch(a, b)
    p0 = (1 + bv[2]) / 2
    p1 = (1 - bv[2]) / 2
    print(f"{label:<8} {bv[2]:>6.3f} {p0:>10.4f} {p1:>10.4f}")


## Summary

| Concept | Key point |
|---|---|
| **Spinor → Bloch** | $(x,y,z) = (2\,\text{Re}(\alpha^*\beta),\;2\,\text{Im}(\alpha^*\beta),\;|\alpha|^2-|\beta|^2)$ |
| **Global phase** | $e^{i\varphi}|\psi\rangle$ maps to the **same** Bloch point |
| **Antipodes** | Antipodal states are orthogonal |
| **Measurement** | $P(|0\rangle) = (1+z)/2$ |

In `rqm-core`, `Spinor.to_bloch()` handles the conversion.  
This notebook shows the geometry underneath that call.

**Next:** [03_su2_rotations_and_geometry.ipynb](03_su2_rotations_and_geometry.ipynb)
